# 10 — Explicabilidade (Integrated Gradients)
Atribuição por conexão (par de ROIs) para o MLP, agregada por ROI. Linha de base = média do treino (zeros após padronização).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, torch
import config
from nn_utils import MLP

d = np.load(config.ARQ_DATASET, allow_pickle=True)
Xte, yte = d["X_test"], d["y_test"]
sc = joblib.load(config.MODELOS / "scaler_mlp.joblib")
roi_ids = np.load(config.ARQ_ROIS)

Xte_t = torch.tensor(sc.transform(Xte), dtype=torch.float32)
model = MLP(Xte_t.shape[1])
model.load_state_dict(torch.load(config.MODELOS / "mlp.pt", map_location="cpu"))
model.eval()

In [ ]:
def integrated_gradients(model, x, passos=50):
    base = torch.zeros_like(x)
    alfas = torch.linspace(0, 1, passos).view(-1, 1)
    pts = (base + alfas * (x - base)).requires_grad_(True)
    grads = torch.autograd.grad(model(pts).sum(), pts)[0]
    return ((x - base) * grads.mean(0)).detach()

attr = torch.stack([integrated_gradients(model, x) for x in Xte_t]).numpy()
print("Atribuições:", attr.shape)

In [ ]:
iu = np.triu_indices(config.N_ROIS, k=1)
imp = np.abs(attr).mean(0)

top = np.argsort(imp)[::-1][:20]
tabela = pd.DataFrame({
    "ROI_a": roi_ids[iu[0][top]], "ROI_b": roi_ids[iu[1][top]],
    "importancia": imp[top],
    "atrib_media_TDAH": attr[yte == 1][:, top].mean(0),
    "atrib_media_controle": attr[yte == 0][:, top].mean(0),
})
display(tabela.round(4))
tabela.to_csv(config.RESULTADOS / "xai_top_conexoes.csv", index=False)

In [ ]:
M = np.zeros((config.N_ROIS, config.N_ROIS))
M[iu] = imp; M = M + M.T

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
im = ax[0].imshow(M, cmap="viridis"); ax[0].set_title("Importância por conexão")
plt.colorbar(im, ax=ax[0], fraction=0.046)

imp_roi = M.sum(1)
t = np.argsort(imp_roi)[::-1][:15]
ax[1].barh([f"ROI {roi_ids[i]}" for i in t][::-1], imp_roi[t][::-1])
ax[1].set_title("Top 15 ROIs (soma das importâncias)")
plt.tight_layout(); plt.savefig(config.FIGURAS / "10_xai.png", dpi=150); plt.show()

pd.DataFrame({"roi": roi_ids, "importancia": imp_roi}).sort_values("importancia", ascending=False)\
  .to_csv(config.RESULTADOS / "xai_importancia_roi.csv", index=False)

Os índices de ROI são os do atlas Craddock 200 (CC200). Para nomear regiões anatômicas, é preciso cruzar com o atlas correspondente. Atribuições indicam o que o modelo usa, não causalidade biológica.